# Dataset

In [3]:
import pandas as pd

# Colunas do tabuleiro
board_cols = [
    'top_left', 'top_middle', 'top_right',
    'mid_left', 'mid_middle', 'mid_right',
    'bot_left', 'bot_middle', 'bot_right'
 ]

# Lê o novo dataset já com classes finais (incluindo 'Possibilidade de fim de jogo')
df = pd.read_csv('../data/tic_tac_toe_balanceado.csv')

# Exibir as primeiras linhas do dataset
df.head()

,top_left,top_middle,top_right,mid_left,mid_middle,mid_right,bot_left,bot_middle,bot_right,classe_final
0,1,1,-1,-1,1,1,1,-1,-1,Empate
1,1,1,-1,-1,-1,1,1,1,-1,Empate
2,1,-1,1,1,-1,-1,-1,1,1,Empate
3,-1,1,-1,-1,1,1,1,-1,1,Empate
4,-1,1,-1,1,-1,1,1,-1,1,Empate


In [4]:
# Dicionário de mapeamento para converter valores textuais em numéricos
mapeamento = {'x': 1, 'o': -1, 'b': 0}

# Aplica mapeamento apenas quando a coluna ainda está textual
for col in board_cols:
    if df[col].dtype == object:
        df[col] = df[col].map(mapeamento).fillna(df[col])
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# Verificando a transformação
df.head()

,top_left,top_middle,top_right,mid_left,mid_middle,mid_right,bot_left,bot_middle,bot_right,classe_final
0,1,1,-1,-1,1,1,1,-1,-1,Empate
1,1,1,-1,-1,-1,1,1,1,-1,Empate
2,1,-1,1,1,-1,-1,-1,1,1,Empate
3,-1,1,-1,-1,1,1,1,-1,1,Empate
4,-1,1,-1,1,-1,1,1,-1,1,Empate


In [5]:
import numpy as np


def reclassificar(row):
    tab = row[board_cols].values.tolist()
    combos = [
        [0, 1, 2], [3, 4, 5], [6, 7, 8],  # linhas
        [0, 3, 6], [1, 4, 7], [2, 5, 8],  # colunas
        [0, 4, 8], [2, 4, 6]              # diagonais
    ]

    for c in combos:
        if tab[c[0]] == tab[c[1]] == tab[c[2]] == -1:
            return 'O vence'

    for c in combos:
        if tab[c[0]] == tab[c[1]] == tab[c[2]] == 1:
            return 'X vence'

    if 0 not in tab:
        return 'Empate'

    # Existe uma linha com 2 peças iguais e 1 casa vazia
    for c in combos:
        linha = [tab[c[0]], tab[c[1]], tab[c[2]]]
        if linha.count(0) == 1 and (linha.count(1) == 2 or linha.count(-1) == 2):
            return 'Possibilidade de fim de jogo'

    return 'Tem jogo'

# Recalcula a classe com base no tabuleiro para corrigir rótulos inconsistentes
df['classe_final'] = df.apply(reclassificar, axis=1)

print('Distribuição das classes após tratamento:')
print(df['classe_final'].value_counts())

Distribuição das classes após tratamento:
classe_final
O vence                         200
X vence                         200
Tem jogo                        139
Possibilidade de fim de jogo     61
Empate                           16
Name: count, dtype: int64


In [6]:
# Reforça classes minoritárias até 200 exemplos com reamostragem (com reposição).
target_por_classe = 200

classes_presentes = df['classe_final'].value_counts()
for classe, qtd_atual in classes_presentes.items():
    if qtd_atual < target_por_classe:
        faltam = target_por_classe - qtd_atual
        amostras_extra = df[df['classe_final'] == classe].sample(
            n=faltam, replace=True, random_state=42
        )
        df = pd.concat([df, amostras_extra], ignore_index=True)

print('Distribuição após reforço das classes minoritárias:')
print(df['classe_final'].value_counts())

Distribuição após reforço das classes minoritárias:
classe_final
Empate                          200
O vence                         200
Tem jogo                        200
Possibilidade de fim de jogo    200
X vence                         200
Name: count, dtype: int64


In [7]:
# Realiza o balanceamento final do dataset, garantindo que cada classe (se possível) tenha 200 amostras.
# Isso ajuda a prevenir que o modelo seja viesado.

# Cria uma lista para armazenar os DataFrames balanceados de cada grupo.
balanced_frames = []
# Agrupa o DataFrame pela coluna 'classe_final' e itera sobre cada grupo.
for name, group in df.groupby('classe_final'):
    # Amostra no máximo 200 exemplos de cada grupo. Se o grupo tiver menos de 200, ele pega todos os exemplos.
    # 'random_state=42' garante que a amostragem seja reprodutível.
    balanced_frames.append(group.sample(min(len(group), 200), random_state=42))
# Concatena todos os DataFrames amostrados em um único DataFrame balanceado e reseta o índice.
df_balanceado = pd.concat(balanced_frames).reset_index(drop=True)

# Garante que todas as colunas de posição do tabuleiro são numéricas (int).
# Define as colunas que representam as posições do tabuleiro.
board_cols = ['top_left', 'top_middle', 'top_right', 'mid_left', 'mid_middle', 'mid_right', 'bot_left', 'bot_middle', 'bot_right']
# Itera sobre cada coluna do tabuleiro.
for col in board_cols:
    df_balanceado[col] = pd.to_numeric(df_balanceado[col], errors='coerce').fillna(0).astype(int)

# Remove a coluna original 'classe' se ela ainda existir, pois 'classe_final' é a coluna alvo para o modelo.
if 'classe' in df_balanceado.columns:
    df_balanceado = df_balanceado.drop('classe', axis=1)

# Imprime a distribuição final das classes no dataset balanceado para confirmar o resultado do balanceamento.
print("Distribuição final das classes no dataset balanceado (200 amostras por classe, quando possível):")
print(df_balanceado['classe_final'].value_counts())

Distribuição final das classes no dataset balanceado (200 amostras por classe, quando possível):
classe_final
Empate                          200
O vence                         200
Possibilidade de fim de jogo    200
Tem jogo                        200
X vence                         200
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
# Este bloco de código implementa um algoritmo de classificação de Árvore de Decisão.

# 1. Separar entrada (X) e saída (y).
# X (features/variáveis independentes): As 9 primeiras colunas, que representam o estado do tabuleiro.
X = df_balanceado.iloc[:, :9]
# y (target/variável dependente): A coluna 'classe_final', que contém o resultado do jogo.
y = df_balanceado['classe_final']

# 2. Divisão do Dataset em conjuntos de treino e teste.
# O dataset é dividido em 80% para treino e 20% para teste.
# 'test_size=0.2' especifica que 20% dos dados serão usados para teste.
# 'random_state=42' garante a reprodutibilidade da divisão.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Criar e treinar a Árvore de Decisão.
# 'DecisionTreeClassifier' é o modelo de árvore de decisão.
# 'criterion='entropy'' usa a entropia para medir a qualidade da divisão, buscando maximizar o ganho de informação.
# 'max_depth=None' permite que os nós sejam expandidos até que todas as folhas sejam puras ou contenham menos que 'min_samples_split' amostras.
modelo_arvore = DecisionTreeClassifier(criterion='entropy', max_depth=None)
modelo_arvore.fit(X_train, y_train)

# 4. Avaliação de Resultados.
# Usa o modelo treinado para fazer predições no conjunto de teste (dados que o modelo nunca viu antes).
predicoes = modelo_arvore.predict(X_test)
print("Relatório de Classificação - Árvore de Decisão:")
print(classification_report(y_test, predicoes))

Relatório de Classificação - Árvore de Decisão:
                              precision    recall  f1-score   support

                      Empate       0.87      1.00      0.93        33
                     O vence       0.86      0.67      0.75        48
Possibilidade de fim de jogo       0.79      1.00      0.88        37
                    Tem jogo       0.95      1.00      0.97        38
                     X vence       0.79      0.68      0.73        44

                    accuracy                           0.85       200
                   macro avg       0.85      0.87      0.85       200
                weighted avg       0.85      0.85      0.84       200



In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Este bloco de código implementa um algoritmo de classificação Random Forest.

# 1. Criar e treinar o modelo Random Forest.
# 'n_estimators' é o número de árvores na floresta (geralmente, quanto mais, melhor, mas aumenta o tempo de computação).
# 'random_state=42' garante a reprodutibilidade.
modelo_random_forest = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_random_forest.fit(X_train, y_train)

# 2. Avaliação de Resultados.
# Usa o modelo treinado para fazer predições no conjunto de teste.
predicoes_rf = modelo_random_forest.predict(X_test)
# Imprime um relatório de classificação detalhado para o modelo Random Forest.
print("Relatório de Classificação - Random Forest:")
print(classification_report(y_test, predicoes_rf))

Relatório de Classificação - Random Forest:
                              precision    recall  f1-score   support

                      Empate       1.00      1.00      1.00        33
                     O vence       0.92      0.98      0.95        48
Possibilidade de fim de jogo       0.97      0.97      0.97        37
                    Tem jogo       1.00      0.97      0.99        38
                     X vence       0.98      0.93      0.95        44

                    accuracy                           0.97       200
                   macro avg       0.97      0.97      0.97       200
                weighted avg       0.97      0.97      0.97       200



In [10]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

# Este bloco de código implementa um algoritmo de classificação Multi-Layer Perceptron (MLP/Rede Neural Artificial).

# 1. Criar e treinar o modelo MLP.
# 'hidden_layer_sizes' define a arquitetura da rede: (12,8) significa 2 camadas ocultas com 100 e 50 neurônios.
# 'activation='relu'' usa a função de ativação ReLU (Rectified Linear Unit).
# 'solver='adam'' é um otimizador eficiente para treinamento.
# 'max_iter=1000' define o número máximo de iterações.
# 'random_state=42' garante a reprodutibilidade.
modelo_mlp = MLPClassifier(hidden_layer_sizes=(7), max_iter=1000, activation='relu', solver='adam')
modelo_mlp.fit(X_train, y_train)

# 2. Avaliação de Resultados.
# Usa o modelo treinado para fazer predições no conjunto de teste.
predicoes_mlp = modelo_mlp.predict(X_test)
# Imprime um relatório de classificação detalhado para o modelo MLP.
print("Relatório de Classificação - Multi-Layer Perceptron (MLP):")
print(classification_report(y_test, predicoes_mlp))

Relatório de Classificação - Multi-Layer Perceptron (MLP):
                              precision    recall  f1-score   support

                      Empate       0.92      1.00      0.96        33
                     O vence       0.84      0.77      0.80        48
Possibilidade de fim de jogo       0.50      0.38      0.43        37
                    Tem jogo       0.63      0.68      0.66        38
                     X vence       0.73      0.84      0.78        44

                    accuracy                           0.73       200
                   macro avg       0.72      0.73      0.73       200
                weighted avg       0.73      0.73      0.73       200



/Users/joaodalagnol/Documents/projetos/Jogo_Da_Velha_T1_IA/.venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


### Salvando o Dataset Balanceado e os Modelos Treinados



In [ ]:
import joblib

# 1. Salvar o dataset balanceado em CSV
output_csv_path = '../data/tic_tac_toe_balanceado.csv'
df_balanceado.to_csv(output_csv_path, index=False)
print(f"Dataset balanceado salvo em: {output_csv_path}")

# 2. Salvar o modelo de Árvore de Decisão
output_dt_model_path = '../models/modelo_arvore_decisao.joblib'
joblib.dump(modelo_arvore, output_dt_model_path)
print(f"Modelo de Árvore de Decisão salvo em: {output_dt_model_path}")

# 3. Salvar o modelo Random Forest
output_rf_model_path = '../models/modelo_random_forest.joblib'
joblib.dump(modelo_random_forest, output_rf_model_path)
print(f"Modelo Random Forest salvo em: {output_rf_model_path}")


Dataset balanceado salvo em: ../models/tic_tac_toe_balanceado.csv
Modelo de Árvore de Decisão salvo em: ../models/modelo_arvore_decisao.joblib
Modelo Random Forest salvo em: ../models/modelo_random_forest.joblib


### Código para o Algoritmo K-NN

In [12]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# 1. Configurar o K-NN e os parâmetros para a Validação Cruzada (GridSearch)
knn = KNeighborsClassifier()

# Testar diferentes números de vizinhos (ímpares para evitar empate) e métricas de distância
parametros_knn = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'metric': ['euclidean', 'manhattan']
}

# O GridSearchCV faz a validação cruzada para encontrar a melhor configuração
grid_knn = GridSearchCV(knn, parametros_knn, cv=5, scoring='accuracy')

# 2. Treinar o modelo
print("Treinando K-NN...")
grid_knn.fit(X_train, y_train)

# Melhor configuração encontrada
melhor_knn = grid_knn.best_estimator_
print(f"Melhores parâmetros K-NN: {grid_knn.best_params_}\\n")

# 3. Avaliação de Resultados
predicoes_knn = melhor_knn.predict(X_test)
print("Relatório de Classificação - K-Nearest Neighbors (K-NN):")
print(classification_report(y_test, predicoes_knn))

Treinando K-NN...
Melhores parâmetros K-NN: {'metric': 'euclidean', 'n_neighbors': 3}\n
Relatório de Classificação - K-Nearest Neighbors (K-NN):
                              precision    recall  f1-score   support

                      Empate       1.00      1.00      1.00        33
                     O vence       0.91      0.88      0.89        48
Possibilidade de fim de jogo       0.78      0.95      0.85        37
                    Tem jogo       0.97      0.95      0.96        38
                     X vence       0.97      0.86      0.92        44

                    accuracy                           0.92       200
                   macro avg       0.93      0.93      0.92       200
                weighted avg       0.93      0.92      0.92       200



### Código para o Algoritmo SVM

In [13]:
from sklearn.svm import SVC

# 1. Configurar o SVM e os parâmetros para a Validação Cruzada
svm = SVC(random_state=42)

# Testar diferentes kernels (linear vs não-linear) e a força da regularização (C)
parametros_svm = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf']
}

# Validação cruzada com GridSearch
grid_svm = GridSearchCV(svm, parametros_svm, cv=5, scoring='accuracy')

# 2. Treinar o modelo
print("Treinando SVM...")
grid_svm.fit(X_train, y_train)

# Melhor configuração encontrada
melhor_svm = grid_svm.best_estimator_
print(f"Melhores parâmetros SVM: {grid_svm.best_params_}\\n")

# 3. Avaliação de Resultados
predicoes_svm = melhor_svm.predict(X_test)
print("Relatório de Classificação - Support Vector Machine (SVM):")
print(classification_report(y_test, predicoes_svm))

Treinando SVM...
Melhores parâmetros SVM: {'C': 10, 'kernel': 'rbf'}\n
Relatório de Classificação - Support Vector Machine (SVM):
                              precision    recall  f1-score   support

                      Empate       1.00      1.00      1.00        33
                     O vence       1.00      1.00      1.00        48
Possibilidade de fim de jogo       0.95      1.00      0.97        37
                    Tem jogo       1.00      0.95      0.97        38
                     X vence       1.00      1.00      1.00        44

                    accuracy                           0.99       200
                   macro avg       0.99      0.99      0.99       200
                weighted avg       0.99      0.99      0.99       200



### Código para Salvar os Modelos K-NN e SVM

In [14]:
import joblib

# Exportar o melhor K-NN
output_knn_model_path = '../models/modelo_knn.joblib'
joblib.dump(melhor_knn, output_knn_model_path)
print(f"Modelo K-NN salvo em: {output_knn_model_path}")

# Exportar o melhor SVM
output_svm_model_path = '../models/modelo_svm.joblib'
joblib.dump(melhor_svm, output_svm_model_path)
print(f"Modelo SVM salvo em: {output_svm_model_path}")

Modelo K-NN salvo em: ../models/modelo_knn.joblib
Modelo SVM salvo em: ../models/modelo_svm.joblib
